In [1]:
import pandas

In [2]:
# Cell 1: Import Libraries and Set Display Options
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("=" * 70)
print("FORECAST vs ACTUALS - WMAPE & VARIANCE ANALYSIS")
print("=" * 70)

FORECAST vs ACTUALS - WMAPE & VARIANCE ANALYSIS


In [3]:
# data_path_prediction=r'D:\PivotX_advisors\Abhishek_csvs\forecast_nov2025_jan2026_L5_new.csv'
# data_path_prediction=r'D:\PivotX_advisors\level5\forecast_14months_level5_combined.csv'
# data_path_prediction=r"C:\Users\rohan\Downloads\forecasts_3months 3 (1).csv"
data_path_prediction= r"D:\PivotX_advisors\Abhishek_csvs\forecasts_3months_reconciled_to_direct_level2.csv"
df_forecast = pd.read_csv(data_path_prediction)

print(f"Dataset Shape: {df_forecast.shape[0]:,} rows × {df_forecast.shape[1]} columns")
# print(f"Date Range: {df_forecast['date'].min()} to {df_forecast['date'].max()}")
print(f"\nColumns: {df_forecast.columns.tolist()}")
print(f"\nData Types:\n{df_forecast.dtypes}")
# df_forecast.head()

Dataset Shape: 2,862 rows × 21 columns

Columns: ['opcos', 'entity_region', 'entity_country', 'entity_name', 'account_name', 'level2', 'level5', 'date', 'forecast_horizon', 'occur_prob', 'forecast_amount', 'forecast_amount_uncapped', 'cap_abs', 'zero_ratio_hist', 'hist_variance', 'method', 'bottom_parent_total', 'direct_level2_total', 'recon_factor', 'recon_factor_raw', 'forecast_amount_before_recon']

Data Types:
opcos                            object
entity_region                    object
entity_country                   object
entity_name                      object
account_name                     object
level2                           object
level5                           object
date                             object
forecast_horizon                  int64
occur_prob                      float64
forecast_amount                 float64
forecast_amount_uncapped        float64
cap_abs                         float64
zero_ratio_hist                 float64
hist_variance         

In [4]:
df_forecast['date'] = pd.to_datetime(df_forecast['date'], errors='coerce')
df_forecast['month'] = df_forecast['date'].dt.to_period('M')

df_forecast_monthly = (
    df_forecast.groupby(
        ['month', 'opcos', 'entity_region','entity_country',
         'entity_name','account_name','level2','level5'],
        as_index=False
    ).agg({
        'forecast_amount': 'sum',
        'forecast_amount_uncapped': 'sum',
        # 'predicted_amount_raw': 'sum',   # or 'mean' depending on logic
        # 'occur_prob': 'mean'             # probability → usually mean
    })
)

print(df_forecast_monthly)

        month opcos entity_region  entity_country  \
0     2026-02   HDS          AMER          Canada   
1     2026-02   HDS          AMER          Canada   
2     2026-02   HDS          AMER          Canada   
3     2026-02   HDS          AMER          Canada   
4     2026-02   HDS          AMER          Canada   
...       ...   ...           ...             ...   
2857  2026-04    HV          EMEA  United Kingdom   
2858  2026-04    HV          EMEA  United Kingdom   
2859  2026-04    HV          EMEA  United Kingdom   
2860  2026-04    HV          EMEA  United Kingdom   
2861  2026-04    HV          EMEA  United Kingdom   

                                     entity_name account_name  \
0     Hitachi Digital Services Canada Inc. 00190   4000011853   
1     Hitachi Digital Services Canada Inc. 00190   4000011853   
2     Hitachi Digital Services Canada Inc. 00190   4000011853   
3     Hitachi Digital Services Canada Inc. 00190   4000011853   
4     Hitachi Digital Services Canada 

In [4]:
# import pandas as pd

# # Ensure date column is datetime
# df_forecast['date'] = pd.to_datetime(df_forecast['date'], errors='coerce')
# # Add level2 mapping
# # df_forecast['level2'] = df_forecast['level5'].map(level2_map)

# # Create year-month period column
# df_forecast['month'] = df_forecast['date'].dt.to_period('M')

# # Group by monthly level
# df_forecast_monthly = (
#     df_forecast.groupby(['month', 'opcos', 'entity_region','entity_country','entity_name','account_name','level2','level5'], as_index=False)
#     ['forecast_amount']
#     .sum()
# )

# print(df_forecast_monthly)

In [79]:
# df_forecast_monthly[:100]

In [5]:
# Cell 2: Load Data
data_path_1 = r'D:\PivotX_advisors\level5\bquxjob_eb6c43c_19d39d13f8f.csv'
df_actual_1 = pd.read_csv(data_path_1)

print(f"Dataset Shape: {df_actual_1.shape[0]:,} rows × {df_actual_1.shape[1]} columns")
# print(f"Date Range: {df_actual['date'].min()} to {df_actual['date'].max()}")
print(f"\nColumns: {df_actual_1.columns.tolist()}")
print(f"\nData Types:\n{df_actual_1.dtypes}")
# df_actual.head()

Dataset Shape: 4,631 rows × 45 columns

Columns: ['flow_category', 'bank_branch_id', 'source', 'instance', 'account', 'account_business_id', 'id', 'source_primary_key', 'source_file_name', 'account_identifier', 'transaction_currency', 'transaction_amount', 'value_datetime', 'booking_datetime', 'description', 'extended_details', 'canonical_update_timestamp', 'account_name', 'account_alternative_name', 'account_holder', 'account_holder_id', 'account_class', 'cash_pool_role', 'cash_pool_name', 'bank_name', 'account_currency', 'entity_country', 'entity_name', 'entity_alternative_name', 'entity_code', 'entity_currency', 'entity_region', 'entity_top_level', 'branch_name', 'level1', 'level2', 'level3', 'level4', 'level5', 'fx_rate', 'transaction_amount_usd', 'forecast_flag', 'opcos', 'period_month', 'date']

Data Types:
flow_category                  object
bank_branch_id                 object
source                         object
instance                       object
account                

In [6]:
columns_to_keep =['opcos','value_datetime','entity_region','entity_country','entity_name','account_name','level2','level5','transaction_amount_usd']
df_actual_1=df_actual_1[df_actual_1['source']=='OAT']
df_actual_1=df_actual_1[columns_to_keep]
df_actual_1.isnull().sum()

opcos                     7
value_datetime            0
entity_region             0
entity_country            0
entity_name               0
account_name              0
level2                    0
level5                    0
transaction_amount_usd    0
dtype: int64

In [7]:
df_actual_1=df_actual_1.dropna()
df_actual_1=df_actual_1.rename(columns={'transaction_amount_usd':'actual_amount','value_datetime':'date'})

In [8]:
import pandas as pd

# Ensure date column is datetime
df_actual_1['date'] = pd.to_datetime(df_actual_1['date'], errors='coerce')
# Add level2 mapping
# df_forecast['level2'] = df_forecast['level5'].map(level2_map)

# Create year-month period column
df_actual_1['month'] = df_actual_1['date'].dt.to_period('M')

# Group by monthly level
df_actual_monthly = (
    df_actual_1.groupby(['month', 'opcos', 'entity_region','entity_country','entity_name','account_name','level2','level5'], as_index=False)
    ['actual_amount']
    .sum()
)

print(df_actual_monthly)

        month opcos entity_region  entity_country  \
0     2026-02   HDS          AMER          Canada   
1     2026-02   HDS          AMER          Canada   
2     2026-02   HDS          AMER          Canada   
3     2026-02   HDS          AMER          Canada   
4     2026-02   HDS          AMER          Canada   
...       ...   ...           ...             ...   
1550  2026-03    HV          EMEA  United Kingdom   
1551  2026-03    HV          EMEA  United Kingdom   
1552  2026-03    HV          EMEA  United Kingdom   
1553  2026-03    HV          EMEA  United Kingdom   
1554  2026-03    HV          EMEA  United Kingdom   

                                     entity_name account_name  \
0     Hitachi Digital Services Canada Inc. 00190   4000011853   
1     Hitachi Digital Services Canada Inc. 00190   4000011853   
2     Hitachi Digital Services Canada Inc. 00190   4000011853   
3     Hitachi Digital Services Canada Inc. 00190   4000011853   
4     Hitachi Digital Services Canada 

In [9]:
df_actual_monthly.duplicated().sum()

0

In [10]:
# Cell 2: Load Data
data_path_2 = r'D:\PivotX_advisors\level5\Grid (Forecast Worksheet)_20260330_112016.csv'
df_actual_2 = pd.read_csv(data_path_2)

print(f"Dataset Shape: {df_actual_2.shape[0]:,} rows × {df_actual_2.shape[1]} columns")
# print(f"Date Range: {df_actual['date'].min()} to {df_actual['date'].max()}")
print(f"\nColumns: {df_actual_2.columns.tolist()}")
print(f"\nData Types:\n{df_actual_2.dtypes}")
# df_actual.head()

Dataset Shape: 1,208 rows × 17 columns

Columns: ['Account Name', 'Level 2', 'Level 5', 'Type', 'Mar 2026', 'Apr 2026', 'May 2026', 'Jun 2026', 'Jul 2026', 'Aug 2026', 'Sep 2026', 'Oct 2026', 'Nov 2026', 'Dec 2026', 'Jan 2027', 'Feb 2027', 'Mar 2027']

Data Types:
Account Name     object
Level 2          object
Level 5          object
Type             object
Mar 2026        float64
Apr 2026        float64
May 2026        float64
Jun 2026        float64
Jul 2026        float64
Aug 2026        float64
Sep 2026        float64
Oct 2026        float64
Nov 2026        float64
Dec 2026        float64
Jan 2027        float64
Feb 2027        float64
Mar 2027        float64
dtype: object


In [11]:
import pandas as pd

# 1. Identify which columns are dates and which are descriptive
# We want to keep Account Name, Level 2, Level 5, AND Type as static columns
id_cols = ['Account Name', 'Level 2', 'Level 5', 'Type']

# 2. Melt the dataframe
# This tells Pandas: "Keep these 4 columns as they are, and turn everything else into rows"
df_long = df_actual_2.melt(
    id_vars=id_cols, 
    var_name='Month', 
    value_name='Amount'
)

# 3. Clean up the 'Month' column
# Sometimes 'melt' catches extra header junk; we remove anything that isn't a date string
# We only keep rows where 'Month' actually looks like a date (e.g., contains '202')
df_long = df_long[df_long['Month'].str.contains('202', na=False)]

# 4. Apply your Level filters
df_clean = df_long.dropna(subset=['Level 2', 'Level 5']).copy()

# 5. Now convert to datetime (this should work without errors now)
df_clean['Month'] = pd.to_datetime(df_clean['Month'], format='%b %Y')

# 6. Final Sort
df_clean = df_clean.sort_values(['Account Name', 'Month'])

print(df_clean.head())

         Account Name         Level 2                      Level 5     Type  \
2     0-051-0045665-8  Free Cash Flow  Miscellaneous Disbursements  Netflow   
1210  0-051-0045665-8  Free Cash Flow  Miscellaneous Disbursements  Netflow   
2418  0-051-0045665-8  Free Cash Flow  Miscellaneous Disbursements  Netflow   
3626  0-051-0045665-8  Free Cash Flow  Miscellaneous Disbursements  Netflow   
4834  0-051-0045665-8  Free Cash Flow  Miscellaneous Disbursements  Netflow   

          Month     Amount  
2    2026-03-01 -13,305.00  
1210 2026-04-01       0.00  
2418 2026-05-01       0.00  
3626 2026-06-01       0.00  
4834 2026-07-01       0.00  


In [12]:
df_clean['Month'] = pd.to_datetime(df_clean['Month'], format='%b %Y', errors='coerce')
df_clean = df_clean.dropna(subset=['Month'])

In [13]:
# Filter out rows where the month is March (3)
df_clean = df_clean[df_clean['Month'].dt.month != 3]

# Verify the result
print(df_clean['Month'].unique())

<DatetimeArray>
['2026-04-01 00:00:00', '2026-05-01 00:00:00', '2026-06-01 00:00:00',
 '2026-07-01 00:00:00', '2026-08-01 00:00:00', '2026-09-01 00:00:00',
 '2026-10-01 00:00:00', '2026-11-01 00:00:00', '2026-12-01 00:00:00',
 '2027-01-01 00:00:00', '2027-02-01 00:00:00']
Length: 11, dtype: datetime64[ns]


In [14]:
# Rename columns
df_clean = df_clean.rename(columns={
        # "Region": "entity_region",
        # "Country": "entity_country",
    "Account Name": "account_name",
    "Level 2": "level2",
    "Level 5": "level5",
    "Amount": "actual_amount"
})

# Step 1: Drop old region
# df_long = df_long.drop(columns=["entity_country"])
df_clean=df_clean.dropna()
# Step 2: Create mapping table (unique values only)
mapping_cols = ["account_name",  "opcos", 'entity_region','entity_country','entity_name']

df_map = df_actual_1[mapping_cols].drop_duplicates()

# Step 3: Merge (LEFT JOIN)
df_clean = df_clean.merge(
    df_map,
    on=["account_name"],   # adjust if needed
    how="left"
)
# df_clean['level2'] = df_clean['level5'].map(level2_map)
# Step 4: Reorder columns (optional)
df_clean = df_clean[
    ["Month", "opcos", "entity_region",'entity_country','entity_name','account_name','level2', 'level5',"actual_amount"]
]
df_clean.head()

,Month,opcos,entity_region,entity_country,entity_name,account_name,level2,level5,actual_amount
0,2026-04-01,HV,AMER,Chile,HITACHI VANTARA (CHILE) LIMITADA 330,0-051-0045665-8,Free Cash Flow,Miscellaneous Disbursements,0.00
1,2026-05-01,HV,AMER,Chile,HITACHI VANTARA (CHILE) LIMITADA 330,0-051-0045665-8,Free Cash Flow,Miscellaneous Disbursements,0.00
2,2026-06-01,HV,AMER,Chile,HITACHI VANTARA (CHILE) LIMITADA 330,0-051-0045665-8,Free Cash Flow,Miscellaneous Disbursements,0.00
3,2026-07-01,HV,AMER,Chile,HITACHI VANTARA (CHILE) LIMITADA 330,0-051-0045665-8,Free Cash Flow,Miscellaneous Disbursements,0.00
4,2026-08-01,HV,AMER,Chile,HITACHI VANTARA (CHILE) LIMITADA 330,0-051-0045665-8,Free Cash Flow,Miscellaneous Disbursements,0.00


In [15]:
import pandas as pd

# Ensure date column is datetime
# df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')
# Add level2 mapping
# df_forecast['level2'] = df_forecast['level5'].map(level2_map)

# Create year-month period column
df_clean['month'] = df_clean['Month'].dt.to_period('M')

# Group by monthly level
df_actual_monthly_2 = (
    df_clean.groupby(['month', 'opcos', 'entity_region','entity_country','entity_name','account_name','level2','level5'], as_index=False)
    ['actual_amount']
    .sum()
)

print(df_actual_monthly_2)

        month opcos entity_region  entity_country  \
0     2026-04   HDS          AMER          Canada   
1     2026-04   HDS          AMER          Canada   
2     2026-04   HDS          AMER          Canada   
3     2026-04   HDS          AMER          Canada   
4     2026-04   HDS          AMER          Canada   
...       ...   ...           ...             ...   
7827  2027-02    HV          EMEA  United Kingdom   
7828  2027-02    HV          EMEA  United Kingdom   
7829  2027-02    HV          EMEA  United Kingdom   
7830  2027-02    HV          EMEA  United Kingdom   
7831  2027-02    HV          EMEA  United Kingdom   

                                     entity_name account_name  \
0     Hitachi Digital Services Canada Inc. 00190   4000011853   
1     Hitachi Digital Services Canada Inc. 00190   4000011853   
2     Hitachi Digital Services Canada Inc. 00190   4000011853   
3     Hitachi Digital Services Canada Inc. 00190   4000011853   
4     Hitachi Digital Services Canada 

In [16]:
df_actual_monthly_2.duplicated().sum()

0

In [17]:
df_actual_monthly=pd.concat([df_actual_monthly, df_actual_monthly_2], axis=0)
print(df_actual_monthly)

        month opcos entity_region  entity_country  \
0     2026-02   HDS          AMER          Canada   
1     2026-02   HDS          AMER          Canada   
2     2026-02   HDS          AMER          Canada   
3     2026-02   HDS          AMER          Canada   
4     2026-02   HDS          AMER          Canada   
...       ...   ...           ...             ...   
7827  2027-02    HV          EMEA  United Kingdom   
7828  2027-02    HV          EMEA  United Kingdom   
7829  2027-02    HV          EMEA  United Kingdom   
7830  2027-02    HV          EMEA  United Kingdom   
7831  2027-02    HV          EMEA  United Kingdom   

                                     entity_name account_name  \
0     Hitachi Digital Services Canada Inc. 00190   4000011853   
1     Hitachi Digital Services Canada Inc. 00190   4000011853   
2     Hitachi Digital Services Canada Inc. 00190   4000011853   
3     Hitachi Digital Services Canada Inc. 00190   4000011853   
4     Hitachi Digital Services Canada 

In [18]:
df_actual_monthly.duplicated().sum()
# print(df_actual_monthly)

0

In [19]:
print(len(df_actual_monthly))
print(len(df_forecast_monthly))

9387
2862


In [18]:
df_new = df_actual_monthly.merge(
    df_forecast_monthly[['month', 'opcos', 'entity_region','entity_country','entity_name','account_name','level2','level5','forecast_amount','forecast_amount_uncapped']],
    on=['month', 'opcos', 'entity_region','entity_country','entity_name','account_name','level2','level5'],
    how='outer'   # keeps all rows from both dataframes 
)

df_new[:100]

,month,opcos,entity_region,entity_country,entity_name,account_name,level2,level5,actual_amount,forecast_amount,forecast_amount_uncapped
0,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Financing Flows,Increase (Decrease) - Loans from 3rd Party Len...,0.00,NaN,NaN
1,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,A/R - Collections,"520,422.21","234,141.99","709,521.19"
2,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,AP - OPEX Payments,"-168,396.23","-96,183.30","-291,464.54"
3,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,Adjustment to Reconcile Bank Activity to GL,0.00,NaN,NaN
4,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,Miscellaneous Disbursements,0.00,NaN,NaN
5,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,Miscellaneous Receipts,0.00,NaN,NaN
6,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,Payroll Payments,"-278,337.40","-120,661.33","-365,640.39"
7,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,Product Payments - Hitachi Ltd Group,0.00,NaN,NaN
8,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,Related Parties - Payments,0.00,NaN,NaN
9,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,Related Parties - Receipts,0.00,0.00,0.00


In [20]:
df_grouped = df_new.groupby(
    ['month', 'opcos', 'entity_region', 'level2']
).agg({
    'actual_amount': 'sum' ,
    'forecast_amount': 'sum',
    'forecast_amount_uncapped': 'sum'
      # if exists
}).reset_index()

df_grouped[:100]

,month,opcos,entity_region,level2,actual_amount,forecast_amount,forecast_amount_uncapped
0,2026-02,HDS,AMER,Financing Flows,"-5,435,956.83","567,267.68","1,718,992.96"
1,2026-02,HDS,AMER,Free Cash Flow,"14,040,787.02","9,840,936.97","12,648,708.13"
2,2026-02,HDS,AMER,Interco Payments,"-10,107,578.36","-8,126,767.67","-9,947,912.83"
3,2026-02,HDS,AMER,Interco Receipts,"688,703.10","554,399.90","184,799.97"
4,2026-02,HDS,APAC,Financing Flows,"-2,721,355.00","-4,788,542.76","-4,788,542.76"
5,2026-02,HDS,APAC,Free Cash Flow,"-5,043,468.84","-9,165,303.02","-10,889,643.35"
6,2026-02,HDS,APAC,Interco Payments,"-3,515,636.44","-955,350.96","-955,350.96"
7,2026-02,HDS,APAC,Interco Receipts,"10,254,816.60","11,723,648.38","7,554,746.23"
8,2026-02,HDS,EMEA,Financing Flows,-0.00,0.00,0.00
9,2026-02,HDS,EMEA,Free Cash Flow,"2,848,167.17","1,563,516.59","2,793,783.35"


In [21]:
import numpy as np

# add abs_variance1 column after actual_amount
df_new.insert(
    df_new.columns.get_loc('forecast_amount_uncapped') + 1,
    'abs_variance',
    (df_new['actual_amount'] - df_new['forecast_amount']).abs()
)

# denominator = max(|actual|, |forecast|)
denominator = np.maximum(
    df_new['actual_amount'].abs(),
    df_new['forecast_amount'].abs()
)

# pct change
pct_change = (df_new['abs_variance'] / df_new['actual_amount'] ) * 100

# sign rule
sign = np.where(df_new['forecast_amount'] * df_new['actual_amount'] < 0, -1, 1)

df_new.insert(
    loc=df_new.columns.get_loc('abs_variance') + 1,
    column='pct_change',
    value=pct_change
)

df_new.head(1)

# Create accuracy first
accuracy = 100 - df_new['pct_change'].abs()

# Keep within 0–100
accuracy = accuracy.clip(0, 100)

# Insert right after pct_change
df_new.insert(
    loc=df_new.columns.get_loc('pct_change') + 1,
    column='accuracy',
    value=accuracy
)
df_new.head()

,month,opcos,entity_region,entity_country,entity_name,account_name,level2,level5,actual_amount,forecast_amount,forecast_amount_uncapped,abs_variance,pct_change,accuracy
0,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Financing Flows,Increase (Decrease) - Loans from 3rd Party Len...,0.00,NaN,NaN,NaN,NaN,NaN
1,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,A/R - Collections,"520,422.21","234,141.99","709,521.19","286,280.22",55.01,44.99
2,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,AP - OPEX Payments,"-168,396.23","-96,183.30","-291,464.54","72,212.93",-42.88,57.12
3,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,Adjustment to Reconcile Bank Activity to GL,0.00,NaN,NaN,NaN,NaN,NaN
4,2026-02,HDS,AMER,Canada,Hitachi Digital Services Canada Inc. 00190,4000011853,Free Cash Flow,Miscellaneous Disbursements,0.00,NaN,NaN,NaN,NaN,NaN


In [22]:
df_new.drop_duplicates(inplace=True)

In [23]:
df_new.to_csv(r'D:\PivotX_advisors\Abhishek_csvs\comparison_level5_new_recon.csv', index=False)

In [24]:
df_grouped.to_csv(r'D:\PivotX_advisors\Abhishek_csvs\comparison_level2_new_recon.csv', index=False)

In [25]:
#calculate wmape for forecast_amount
wmape = df_new['abs_variance'].sum() / df_new['actual_amount'].abs().sum() * 100
print(f"WMAPE for forecast_amount: {wmape:.2f}%")

WMAPE for forecast_amount: 74.22%
